In [ ]:
# 0) Colab: mount Drive (train-data + runs) + clone code từ GitHub
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

DRIVE_DIR = '/content/drive/MyDrive/recommender_train_colab'   # train-data/ + runs/ sống ở đây
REPO = 'https://github.com/<user>/<repo>.git'                  # <-- ĐIỀN URL repo GitHub
CODE = '/content/recommender'

# clone lần đầu, các lần sau git pull -> luôn lấy code mới nhất sau khi push (không upload Drive)
if os.path.exists(CODE):
    !cd {CODE} && git pull
else:
    !git clone {REPO} {CODE}

# train-data frozen sống trên Drive -> copy về local 1 lần (đọc parquet nhanh, không qua Drive FUSE)
if not os.path.exists(f'{CODE}/train-data'):
    !cp -r "{DRIVE_DIR}/train-data" "{CODE}/train-data"

%cd {CODE}

In [ ]:
# 1) Imports + path (notebook chạy trong project_local/, model/ là package)
# imp shim: vài lib trên Colab (py3.12) còn import 'imp' đã bị gỡ khỏi stdlib
import sys, importlib
sys.modules['imp'] = importlib

# autoreload: tự nạp lại module .py khi sửa (khỏi restart kernel mỗi lần)
%load_ext autoreload
%autoreload 2

import pathlib
HERE = pathlib.Path.cwd()
MODEL_DIR = HERE if HERE.name == 'model' else HERE / 'model'
sys.path.insert(0, str(MODEL_DIR))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import config, data, model as M, loss, metrics, train
print('torch', torch.__version__)

In [ ]:
# 2) Đường dẫn persistence trên Drive (checkpoint + history + leaderboard sống sót disconnect)
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/recommender_train_colab')
VERSION = 'v2'                                # đổi mỗi đợt experiment: v1, v2, v3...
RUNS_DIR = DRIVE / 'runs' / VERSION            # mỗi run: runs/<VERSION>/<run_name>/best.pt + history.json
RESULTS_CSV = DRIVE / 'runs.csv'               # leaderboard CHUNG mọi version (có cột 'version')
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('version', VERSION, '· runs ->', RUNS_DIR)

In [ ]:
# 3) Base config T4 (Colab free: 2 core / 13GB RAM / 15GB VRAM). ckpt_dir set per-run ở helper (cell 5).
base_cfg = config.TwoTowerConfig(
    # model
    d=128,

    # loss
    tau=0.07,
    beta=1.0,

    # train
    lr=1e-3,
    batch_size=8192,       # T4 VRAM dư -> gấp đôi in-batch negative (contrastive khó hơn)
    epochs=1,
    hist_dropout=0.12,
    m_hardneg=5,

    # eval — eval FULL val mỗi eval_every_steps; subset nhỏ thì giảm xuống
    eval_every_steps=500,

    # infra
    num_workers=2,         # đúng 2 core T4: overlap collate/transfer (cổ chai, không phải GPU)
    # subset=200_000,      # bỏ comment để chạy thử nhanh; None = full
)
print('device', base_cfg.device)   # 'cuda' trên T4
base_cfg

In [ ]:
# 4) Build module (sanity-check shape / param count)
spec, logq, item_table, users, mdl = train.build(base_cfg)
n_params = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
print(f'num_items={item_table.num_items}  num_users={users.num_users}  params={n_params:,}')
print('logq', logq.shape, 'finite', int(torch.isfinite(logq).sum()))

In [ ]:
# 5) Helpers thử nghiệm: mỗi run có tên riêng -> checkpoint/history/leaderboard lưu trên Drive
import json, time, dataclasses

def load_run(run_name):
    """Load best.pt của 1 run -> (cfg, ckpt, model, users, logq) sẵn sàng eval/inference.
    Dựng model theo cfg LƯU TRONG checkpoint (gồm mọi override d/tau/...) -> khớp shape state_dict."""
    ckpt = torch.load(RUNS_DIR / run_name / 'best.pt', weights_only=False)  # file kèm cfg -> cần False
    cfg = ckpt['cfg']
    _, logq, _, users, model = train.build(cfg)
    model.load_state_dict(ckpt['model'])
    model.refresh_item_cache()
    return cfg, ckpt, model, users, logq

def eval_run(cfg, model, users, logq):
    """Eval full val + test cold-by-user -> dict phẳng để ghi 1 dòng leaderboard."""
    out = {}
    for split in ['val', 'test']:
        ds = data.ExamplesDataset(cfg.train_data, split)
        q = metrics.group_examples(ds.user_idx, ds.anime_idx)
        m = metrics.evaluate(model, users, q, logq, cfg.eval_ks)
        for k in cfg.eval_ks:
            out[f'{split}_recall@{k}'] = round(float(m[f'recall@{k}']), 4)
            out[f'{split}_ndcg@{k}']   = round(float(m[f'ndcg@{k}']), 4)
    return out

def run_experiment(run_name, **overrides):
    """Clone base_cfg + override hyperparam -> train -> eval -> append runs.csv.
    overrides là field của TwoTowerConfig (vd use_item_id=True, id_dim=128, tau=0.05, epochs=2)."""
    cfg = dataclasses.replace(base_cfg, **overrides)
    cfg.ckpt_dir = RUNS_DIR / run_name
    t0 = time.time()
    model, history = train.fit(cfg)
    secs = time.time() - t0
    # history -> json (plot lại sau / sống sót disconnect)
    (cfg.ckpt_dir / 'history.json').write_text(json.dumps(history, default=float))
    # eval lại từ best.pt (model tốt nhất, không phải model step cuối) rồi ghi 1 dòng leaderboard
    cfg, ckpt, model_e, users_e, logq_e = load_run(run_name)
    row = {
        'run_name': run_name, 'version': VERSION, 'time': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
        'd': cfg.d, 'tau': cfg.tau, 'beta': cfg.beta, 'lr': cfg.lr,
        'batch_size': cfg.batch_size, 'epochs': cfg.epochs,
        'hist_dropout': cfg.hist_dropout, 'm_hardneg': cfg.m_hardneg,
        'use_item_id': cfg.use_item_id, 'id_dim': cfg.id_dim, 'id_dropout': cfg.id_dropout,
        'best_step': ckpt.get('step'), 'secs': round(secs),
        **eval_run(cfg, model_e, users_e, logq_e),
    }
    df = pd.concat([
        pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame(),
        pd.DataFrame([row]),
    ], ignore_index=True)
    df.to_csv(RESULTS_CSV, index=False)
    print(f"\n✓ {run_name}: test recall@100={row['test_recall@100']}  →  {RESULTS_CSV.name}")
    return cfg, history, row

In [ ]:
# 6) Chạy 1 thử nghiệm. Đổi tên + kwargs cho mỗi lần (checkpoint KHÔNG ghi đè nhau).
cfg, history, row = run_experiment('itemid64', use_item_id=True, id_dim=64, id_dropout=0.2)

# Ví dụ các run tiếp theo (uncomment từng dòng):
# cfg, history, row = run_experiment('content_base')                                       # content-only đối chứng trong v2
# cfg, history, row = run_experiment('itemid128',    use_item_id=True, id_dim=128, id_dropout=0.2)
# cfg, history, row = run_experiment('itemid64_d01', use_item_id=True, id_dim=64,  id_dropout=0.1)
# cfg, history, row = run_experiment('itemid64_d03', use_item_id=True, id_dim=64,  id_dropout=0.3)

In [ ]:
# 7) Đường cong training loss (raw + moving-average) — dùng `history` trả từ run_experiment
ls = np.array(history['loss_steps']); lv = np.array(history['loss_vals'])
plt.figure(figsize=(8, 4))
plt.plot(ls, lv, alpha=0.3, label='loss')
w = max(1, len(lv) // 50)
if len(lv) >= w and w > 1:
    ma = np.convolve(lv, np.ones(w) / w, mode='valid')
    plt.plot(ls[w - 1:], ma, label=f'MA({w})')
plt.xlabel('step'); plt.ylabel('InfoNCE loss'); plt.title('Training loss')
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# 8) Đường cong val metric theo step (recall@K, ndcg@K). Retriever -> để ý recall@100.
es = history['eval_steps']; em = history['eval_metrics']
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for k in cfg.eval_ks:
    ax[0].plot(es, [m[f'recall@{k}'] for m in em], marker='o', label=f'recall@{k}')
    ax[1].plot(es, [m[f'ndcg@{k}'] for m in em], marker='o', label=f'ndcg@{k}')
ax[0].set_title('Val recall@K'); ax[0].set_xlabel('step')
ax[1].set_title('Val ndcg@K'); ax[1].set_xlabel('step')
for a in ax: a.legend(); a.grid(alpha=0.3)
plt.show()

In [ ]:
# 9) Leaderboard: so sánh mọi run đã log. Headline retriever = test_recall@100.
pd.read_csv(RESULTS_CSV).sort_values('test_recall@100', ascending=False)

In [ ]:
# 10) Load 1 run đã lưu để soi: eval lại full val/test + chuẩn bị model_e/users_e/logq_e cho cell định tính
RUN = 'bs8192_base'
cfg, ckpt, model_e, users_e, logq_e = load_run(RUN)
res = eval_run(cfg, model_e, users_e, logq_e)
print(f"[{RUN}] best_step={ckpt.get('step')}")
print(f"  val : recall@100={res['val_recall@100']}  ndcg@100={res['val_ndcg@100']}")
print(f"  test: recall@100={res['test_recall@100']}  ndcg@100={res['test_ndcg@100']}   (headline retriever)")

In [ ]:
# 11) Kiểm tra định tính RETRIEVER: vài user test -> recall@100 + xem truth lọt top-100 ở HẠNG nào
#    (retriever lo "relevant có lọt top-100 không", không phải precision@10 kiểu ranker)
#    dùng model_e/users_e/logq_e/cfg từ cell 10. mal_id tra ở https://myanimelist.net/anime/<mal_id>
K = 100
amap = pd.read_parquet(cfg.train_data / 'anime_id_map.parquet')
idx2mal = dict(zip(amap.anime_idx.astype(int), amap.mal_id.astype(int)))
ds = data.ExamplesDataset(cfg.train_data, 'test')
q = metrics.group_examples(ds.user_idx, ds.anime_idx)
cand = torch.isfinite(logq_e)
for uid in list(q.keys())[:5]:
    hist = torch.from_numpy(users_e.history_pad[[uid]]).long().to(cfg.device)
    ub = {
        'history_ids': hist, 'history_mask': hist != 0,
        'gender_id': torch.tensor([users_e.gender_id[uid]]).to(cfg.device),
        'joined_bucket': torch.tensor([users_e.joined_bucket[uid]]).to(cfg.device),
    }
    s = (model_e.encode_users(ub) @ model_e.item_cache.t())[0]
    s[~cand] = -float('inf'); s[hist[0]] = -float('inf')   # bỏ non-candidate + item đã seen
    top = torch.topk(s, K).indices.cpu().numpy()           # [K] đã xếp hạng
    truth = set(q[uid])
    ranks = [r + 1 for r, i in enumerate(top) if int(i) in truth]   # hạng (1-based) của truth lọt top-K
    recall = len(ranks) / len(truth)
    head = [idx2mal[int(i)] for i in top[:5]]              # 5 gợi ý đầu để eyeball
    print(f"user {uid}: |hist|={int((hist[0] != 0).sum())} |truth|={len(truth)}  "
          f"recall@{K}={recall:.2f} ({len(ranks)}/{len(truth)} lọt top{K})  "
          f"hạng truth={ranks}  top5_mal={head}")